# Lesson 3 — State Space & Stability

*ESP2110 Inverted Pendulum Lab*

**Run in Google Colab:** open the notebook, run the **Setup** cell once, then run
cells top-to-bottom. No local files are required.

## Learning objectives
By the end of this notebook you can:
1. Write the linearized cart-pole in **state-space form** `x_dot = A x + B f`, `y = C x`.
2. Read **stability off the eigenvalues** of `A` for the upright and downward equilibria.
3. Connect each eigenvalue to a **time-domain behaviour** (growth, decay, oscillation).
4. State precisely **what control must fix** (move the unstable eigenvalue into the left half-plane).

### Parameters (same plant used throughout the lab)
| Symbol | Meaning | Value |
| --- | --- | --- |
| `m_c` | Cart mass | 0.5 kg |
| `m_p` | Pole mass | 0.2 kg |
| `L` | Pole length | 0.3 m |
| `g` | Gravity | 9.81 m/s^2 |
| `dt` | Sample time | 0.01 s |

In [ ]:
# --- Setup (safe to re-run) ---
try:
    import numpy, scipy, matplotlib  # noqa: F401
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'matplotlib'], check=True)
print('Environment ready.')

---
## State-space form

The linearized model from Lesson 2 is

$$\dot x = A x + B f,\qquad y = C x + D f,$$

with `x = [p, v, theta, omega]`. If every state is measured, `C = I` (4x4) and `D = 0`.
**Stability is governed entirely by the eigenvalues of `A`:** a mode `e^{\lambda t}` decays if
`Re(lambda) < 0`, grows if `Re(lambda) > 0`, and oscillates if `lambda` is imaginary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m_c, m_p, L, g, dt = 0.5, 0.2, 0.3, 9.81, 0.01

# Upright linearization (theta = 0)
A_up = np.array([
    [0, 1, 0,                       0],
    [0, 0, -m_p * g / m_c,          0],
    [0, 0, 0,                       1],
    [0, 0, (m_c + m_p) * g / (L * m_c), 0],
])
B = np.array([0, 1 / m_c, 0, -1 / (L * m_c)])
C_full = np.eye(4)

## Part 1 - Upright stability

Compute the eigenvalues of `A_up` and classify the equilibrium. A pendulum balanced straight
up is the textbook unstable equilibrium — the eigenvalues should say so.

In [ ]:
# TODO: compute and print eigenvalues of A_up, then report the max real part and
#       whether the upright equilibrium is stable.


**Expected output.** `eig(A_up) = {0, 0, +6.766, -6.766}`. The **positive real eigenvalue
+6.766** makes the upright equilibrium a **saddle -> unstable**: any tilt grows like `e^{6.766 t}`
(it e-folds every ~0.15 s, matching the ~0.67 s fall you saw in Lesson 1). The double pole at 0
is the free cart (position/velocity are not pulled back).

## Part 2 - Downward stability

Linearize about the **hanging-down** equilibrium (`theta = pi`). The only change is the sign of
the gravity term `A[3,2]` (gravity now restores the pole). Build `A_down` and find its eigenvalues.

In [ ]:
# TODO: copy A_up, flip the sign of the [3,2] gravity entry to get A_down (downward),
#       and print its eigenvalues and max real part.


**Expected output.** `eig(A_down) = {0, 0, +6.766j, -6.766j}` — **purely imaginary**. The
hanging pole is a **center**: it neither grows nor decays but **oscillates** at
`omega_n = 6.766 rad/s` (period ~0.93 s). With no damping in this idealized model it swings
forever; real friction would pull these eigenvalues slightly into the left half-plane (stable).

## Part 3 - Eigenvalues become motion

Confirm the eigenvalue story in the time domain: simulate the linear model from a small tilt at
each equilibrium. Upright should **diverge**; downward should **oscillate**.

In [ ]:
# TODO: write roll(Amat, x0, T) that Euler-integrates the linear model x_dot=Amat@x and
#       records pole angle. From theta0=5 deg, plot the upright run (diverges) next to the
#       downward run (oscillates).


**Expected output.** Left: the upright angle **grows exponentially** (the `+6.766` mode).
Right: the downward angle **oscillates** at a steady ~0.93 s period (the imaginary pair). The
plots are the eigenvalues made visible.

## Part 4 - Eigenvalues on the complex plane

Plot both equilibria's eigenvalues together. Left half-plane = stable, right half-plane =
unstable, imaginary axis = marginal. This single picture frames the entire control problem.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(eig_up.real, eig_up.imag, s=80, color='crimson', label='upright', zorder=3)
ax.scatter(eig_dn.real, eig_dn.imag, s=80, marker='s', facecolors='none', edgecolors='steelblue', label='downward', zorder=3)
ax.axhline(0, color='k', lw=.5); ax.axvline(0, color='k', lw=.5)
ax.axvspan(0, 8, color='red', alpha=0.06); ax.axvspan(-8, 0, color='green', alpha=0.06)
ax.set_xlim(-8, 8); ax.set_title('Eigenvalues: upright vs downward')
ax.set_xlabel('Real (1/s)'); ax.set_ylabel('Imag (1/s)'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print('The job of control (Lesson 4A+): drag the +6.766 eigenvalue into the left half-plane.')

## Part 5 - Eigenvalues you can watch (animation)

Two poles from the same 5 deg tilt: the **upright** model (red, the `+6.766` mode runs away) and
the **downward** model (blue, the imaginary pair swings forever). The eigenvalues, animated.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def roll(Amat, x0, T=2.0):
    n = int(T / dt); x = np.array(x0, float); TH = np.zeros(n)
    for k in range(n):
        TH[k] = x[2]; x = x + dt * (Amat @ x)
    return TH

TH_up = roll(A_up,   [0, 0, np.radians(5), 0.0])
TH_dn = roll(A_down, [0, 0, np.radians(5), 0.0])
frames = np.arange(0, len(TH_up), 4)
fig, (axU, axD) = plt.subplots(1, 2, figsize=(8, 3))
for ax, ttl in [(axU, 'Upright: runs away'), (axD, 'Downward: oscillates')]:
    ax.set_xlim(-0.5, 0.5); ax.set_ylim(-0.35, 0.35); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(ttl)
poleU, = axU.plot([], [], lw=3, color='crimson'); bobU, = axU.plot([], [], 'o', color='crimson', ms=8)
poleD, = axD.plot([], [], lw=3, color='steelblue'); bobD, = axD.plot([], [], 'o', color='steelblue', ms=8)

def _u(j):
    i = frames[j]
    thu = np.clip(TH_up[i], -np.pi/2, np.pi/2)            # clamp the runaway for display
    txu, tyu = L * np.sin(thu), L * np.cos(thu)
    poleU.set_data([0, txu], [0, tyu]); bobU.set_data([txu], [tyu])
    thd = TH_dn[i]                                        # measured from downward
    txd, tyd = L * np.sin(thd), -L * np.cos(thd)
    poleD.set_data([0, txd], [0, tyd]); bobD.set_data([txd], [tyd])
    return poleU, bobU, poleD, bobD

anim = animation.FuncAnimation(fig, _u, frames=len(frames), interval=70, blit=False)
plt.close(fig); HTML(anim.to_jshtml())

---
## Checkpoints
- `eig(A_up) = {0, 0, +6.766, -6.766}` -> a positive real mode -> **upright is unstable**.
- `eig(A_down) = {0, 0, +-6.766j}` -> purely imaginary -> **downward oscillates** (marginal).
- Time-domain rollouts match: upright diverges, downward oscillates at ~0.93 s period.
- You can state the control goal: **move the unstable eigenvalue into the left half-plane** (Lesson 4A+).

## Common pitfalls
- **Confusing the equilibria.** Only the sign of `A[3,2]` differs; upright is unstable, downward is not.
- **Ignoring the zero eigenvalues.** The double 0 is the un-regulated cart position/velocity — feedback (Lesson 4B) fixes it.
- **Misreading imaginary eigenvalues.** Purely imaginary = sustained oscillation, *not* decay; it takes damping (negative real part) to settle.
- **Forgetting this is the *linear* model.** Eigenvalues describe behaviour near the equilibrium only.